# 07 Style LLM Forward Pass

Goal: load author-level attention-pooled embeddings from notebook 06, inject one into a frozen causal LM, and verify that style conditioning changes logits. No training yet.

In [ ]:
import os
import sys
from pathlib import Path

def find_project_root():
    env_root = os.getenv("VOICE_CLONE_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "voice_clone").exists():
            return candidate
    raise RuntimeError("Could not find project root. Set VOICE_CLONE_ROOT to the repo path.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

import torch
from transformers import AutoTokenizer

from voice_clone.models import StyleConditionedCausalLM, count_parameters

print("ROOT:", ROOT)

In [ ]:
AUTHOR_EMBEDDINGS_PATH = ROOT / "data" / "processed" / "author_style_embeddings.pt"

# Start cheap for mechanics. Change to "microsoft/Phi-3-mini-4k-instruct" once this works.
BASE_MODEL = "distilgpt2"
AUTHOR_ID = None  # None picks the first saved author
MAX_LENGTH = 128

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else None

print("device:", device)
print("author embeddings:", AUTHOR_EMBEDDINGS_PATH)

In [ ]:
payload = torch.load(AUTHOR_EMBEDDINGS_PATH, map_location="cpu")
author_embeddings = payload["author_embeddings"]
author_ids = sorted(author_embeddings)
author_id = AUTHOR_ID or author_ids[0]
style_embedding = author_embeddings[author_id].unsqueeze(0).to(device)
zero_embedding = torch.zeros_like(style_embedding)

print("authors:", author_ids)
print("selected:", author_id)
print("style shape:", tuple(style_embedding.shape))
print("style norm:", style_embedding.norm(dim=-1).item())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = StyleConditionedCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
).to(device)
model.eval()

print(count_parameters(model))

In [ ]:
draft = "The visit was expected to be brief, but the conversation soon revealed more than anyone intended."

encoded = tokenizer(
    draft,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
)
encoded = {key: value.to(device) for key, value in encoded.items()}

with torch.no_grad():
    out_zero = model(**encoded, style_embedding=zero_embedding)
    out_style = model(**encoded, style_embedding=style_embedding)

logit_delta = (out_style.logits - out_zero.logits).abs().mean().item()
last_zero = out_zero.logits[:, -1, :]
last_style = out_style.logits[:, -1, :]

print("mean abs logit delta:", logit_delta)
print("zero top tokens:", tokenizer.batch_decode(last_zero.topk(5).indices[0]))
print("style top tokens:", tokenizer.batch_decode(last_style.topk(5).indices[0]))

assert logit_delta > 0
print("style injection forward pass ok")